# Homework Cohort 2026 Spark - Week 6

In [2]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 13:26:08--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2764:9600:b:20a5:b140:21, 2600:9000:2764:ac00:b:20a5:b140:21, 2600:9000:2764:e200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2764:9600:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.3’

yellow_tripdata_202 100%[===================>]  67,84M  7,84MB/s    in 8,7s    

2026-03-09 13:26:18 (7,80 MB/s) - ‘yellow_tripdata_2025-11.parquet.3’ saved [71134255/71134255]



# The anatomy of a spark job
Now in order to start a spark session we neeed to create an object in which we define some attributes for our spark jobs

* The .master defined the resources that we can reserve in our session ( in our case is local in which means that we will use the local host or local resources and * sign means that we will use all the available resources for our spark job. if we want only 2 cpus to work we define it like local[2]
* The .appName defines the name of our spark job
* The .getOrCreate Check is it exists to override the job or creates a new one

In [8]:
import pyspark
from pyspark.sql import SparkSession

In [10]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 13:26:27 WARN Utils: Your hostname, consta-ROG-Strix-G731GU-G731GU, resolves to a loopback address: 127.0.1.1; using 192.168.18.75 instead (on interface wlo1)
26/03/09 13:26:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 13:26:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [27]:
df = spark.read.option("header","true").parquet('yellow_tripdata_2025-11.parquet')

this is how we read a csv in spark : 
```bash
    df = spark.read.option("header", "true").parquet('yellow_tripdata_2025-11.parquet')
    
```

In [38]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [42]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [111]:
df4 = df.repartition(4)

In [113]:
df4.write.mode("overwrite").parquet("yellow_2025_11")

In [117]:
import os

path = "yellow_2025_11"
files = [f for f in os.listdir(path) if f.endswith(".parquet")]

sizes = [os.path.getsize(os.path.join(path, f)) / (1024*1024) for f in files]

print("Average size (MB):", sum(sizes)/len(sizes))

Average size (MB): 24.409879207611084


In [49]:
df_csv = spark.read \
.option("header","true") \
.schema(df.schema) \
.csv('head.csv')

In [51]:
df_csv.head(5)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=None, trip_distance=1.68, RatecodeID=None, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 49, 7), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 1, 1, 22), passenger_count=None, trip_distance=2.28, RatecodeID=None, store_and_fwd_flag='N', PULocationID=142, DOLocationID=237, payment_type=1, fare_amount=14.2, extra=1.0, mta_tax=0.5, tip_amount=4.99, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.94, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(

In [53]:
df_csv.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|           NULL|         1.68|      NULL|                 N|          43|    

In [57]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [59]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [61]:
df.registerTempTable('yellow_taxi_2025')

/home/consta/anaconda3/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [63]:
spark.sql("""
SELECT * FROM yellow_taxi_2025 LIMIT 10;
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [119]:
spark.sql("""
SELECT COUNT(*) FROM yellow_taxi_2025 WHERE DATE(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [71]:
count = df.count()
print(count)

4181444


In [127]:
spark.sql("""
SELECT MAX(
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600
) AS max_trip_hours
FROM yellow_taxi_2025
""").show()

+-----------------+
|   max_trip_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [91]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 21:38:07--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.227.153.43, 13.227.153.16, 13.227.153.214, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.227.153.43|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12,04K  --.-KB/s    in 0s      

2026-03-09 21:38:07 (106 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [129]:
zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

In [145]:
df.join(zones, df.PULocationID == zones.LocationID) \
  .groupBy("Zone") \
  .count() \
  .orderBy("count") \
  .show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows


In [139]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

